# Lecture: Why GPUs Speed Up Training (Devices, Batches, Precisions)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain device, CUDA, batch size, and throughput in one sentence each, and read the CPU-flat vs GPU-rising plot (§5: GPU 14.5x at batch 2048, 0.67x at batch 32).
- State what FP32 vs BF16 changes (weights 0.814 MB vs 0.407 MB in §6) and what H2D transfer plus `synchronize()` do (§7).
- Name the two small-batch traps — under-utilization and launch overhead — and say what VRAM tells you (§8).

## The big idea

A CPU is a handful of brilliant chefs, each finishing one dish fast. A GPU is a stadium kitchen with thousands of apprentices who are slow individually but unstoppable together — *if* you give them enough identical work. Training a network is exactly that work: every sample in a **batch** (a bundle of examples processed together) goes through the same matrix multiplications. The **device** is where a tensor lives (`cpu` or `cuda`); **CUDA** is NVIDIA's language for talking to the GPU. With a tiny batch of 32, most GPU apprentices stand idle while the kitchen pays fixed costs — launching kernels and trucking data from CPU memory to GPU memory (the **host-to-device transfer**) — so the CPU can tie or win (0.67x below). With a batch of 2048, all apprentices chop at once and the GPU pulls 14.5x ahead. **Throughput** (samples per second) is the scoreboard. Two more words: **precision** (**FP32** spends 4 bytes per number, **BF16** spends 2 — half the memory traffic, barely any accuracy lost here, plus special **Tensor Core** units that crunch half-precision matrices extra fast), and **synchronize** (GPU work is asynchronous — you must wait for the kitchen to finish before stopping the stopwatch, or your timing lies).

In [ ]:
import numpy as np  # bring in numpy for the illustrative throughput numbers
import matplotlib.pyplot as plt  # bring in pyplot for the scaling plot
np.random.seed(3)  # fix the seed (numbers below are fixed; seed keeps the habit)
batches = np.array([32, 256, 2048])  # define 3 batch sizes: tiny, medium, huge (same sweep as the lab)
cpu_sps = np.array([60000.0, 65000.0, 62000.0])  # define CPU samples/sec: flat, few cores to fill
gpu_sps = np.array([40000.0, 300000.0, 900000.0])  # define GPU samples/sec: starved when small, dominant when fed
print('batches:', batches.tolist())  # print the batch sizes on the x-axis
print('CPU samples/sec:', cpu_sps.tolist())  # print the flat CPU throughput numbers
print('GPU samples/sec:', gpu_sps.tolist())  # print the rising GPU throughput numbers
print('GPU speedup at 2048:', round(float(gpu_sps[2] / cpu_sps[2]), 1), 'x')  # print the headline GPU win: 14.5x
print('GPU vs CPU at batch 32:', round(float(gpu_sps[0] / cpu_sps[0]), 2), '(below 1: CPU wins tiny)')  # print 0.67: CPU wins
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.plot(batches, cpu_sps, 'o-', label='CPU (few fast cores)')  # draw the flat CPU line
ax.plot(batches, gpu_sps, 's-', label='GPU (thousands of small cores)')  # draw the rising GPU line
ax.set_xscale('log')  # use log x because batches span 32 to 2048
ax.set_xlabel('batch size (log scale)')  # label x as batch size
ax.set_ylabel('samples/sec (higher is better)')  # label y as throughput
ax.set_title('Anchor: GPU wins big batches; CPU holds on when tiny')  # honest title says this is a sketch
ax.legend()  # show the legend for the two devices
ax.grid(True, which='both', alpha=0.3)  # add a faint grid on both scales
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure inline

## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 says where code runs, §5 how batch size moves the scoreboard, §6 what fewer bits buy, §7 what the stopwatch must include, §8 what small batches waste.

| Term | One-liner | Section |
|---|---|---|
| device | Where a tensor lives and runs: `cpu` or `cuda` | §4 |
| CUDA | NVIDIA's language for talking to the GPU | §4 |
| batch size | Examples processed together per step (lab sweeps 32, 256, 2048) | §5 |
| throughput (samples/sec) | Examples processed per second — the lab's headline metric | §5 |
| FP32 / BF16 | Number formats: 4 vs 2 bytes per value (weights 0.814 vs 0.407 MB) | §6 |
| Tensor Cores | Special GPU units that crunch half-precision matrix math extra fast | §6 |
| H2D transfer | Host-to-device copy of each batch from CPU to GPU memory (a real cost) | §7 |
| `synchronize()` | Block until queued GPU work finishes (launches are async) | §7 |
| under-utilization / launch overhead | Too-small batches idle the GPU while fixed per-kernel costs dominate | §8 |
| VRAM (peak MB) | GPU memory used; BF16 roughly halves it | §8 |

### §4 Device and CUDA — where the code runs

**In one sentence: the device is where a tensor lives and computes (`cpu` = the computer's main processor, `cuda` = the NVIDIA GPU), and CUDA is the language PyTorch uses to send work there — one line, `xb.to('cuda')`, moves a batch across the wire.**

A CPU has a few fast cores that finish sequential work quickly; a GPU has thousands of small cores that finish *identical parallel* work quickly, plus its own fast memory (VRAM) and high-bandwidth pipes (HBM). `torch.cuda.is_available()` reports whether a GPU exists — on machines without one it is `False` and the lab skips GPU rows honestly instead of faking them. The lab builds the model directly on the target device and dtype (`MLP(...).to(device).to(dtype)`) so the measurement matches reality. Watch it: the diagram traces one batch from CPU memory across the H2D wire to GPU cores and back, with the numbers showing what moves.

In [ ]:
import numpy as np  # reuse numpy for the device-side byte math
import matplotlib.pyplot as plt  # reuse pyplot for the where-code-runs diagram
np.random.seed(1)  # fix the seed (numbers below are fixed; seed keeps the habit)
batch, feats = 256, 784  # define one batch: 256 samples of 784 pixels (MNIST-shaped)
mb_fp32 = batch * feats * 4 / 1e6  # one batch in FP32 megabytes: 256*784*4 bytes
print('one batch (256x784) FP32 MB:', round(float(mb_fp32), 3))  # print 0.803: what crosses the H2D wire per step
print("device rule: tensors compute where they LIVE; xb.to('cuda') moves them")  # print the golden rule
print('CUDA check: torch.cuda.is_available() is False here -> lab skips GPU rows')  # print the honest-skip rule
fig, ax = plt.subplots(figsize=(8, 2.6))  # make a wide flat figure for the device diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix x-range for box placement
ax.set_ylim(0, 3)  # fix y-range for box placement
boxes = [('CPU mem\nhost batch', 1.5), ('H2D wire\ncopy §7', 4.5), ('GPU VRAM\ndevice batch', 7.5), ('GPU cores\ncompute §5', 10.5)]  # stage names along the trip
for name, xc in boxes:  # draw each stage box left to right
    ax.text(xc, 1.5, name, ha='center', va='center', fontsize=9,  # centered stage label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # rounded blue box per station
for xa, xb in [(2.5, 3.5), (5.5, 6.5), (8.6, 9.6)]:  # gaps between consecutive boxes
    ax.annotate('', xy=(xb, 1.5), xytext=(xa, 1.5),  # draw the trip arrow left to right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.set_title('Where code runs: host batch -> H2D wire -> VRAM -> thousands of cores')  # title the trip
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §5 Batch size and throughput — feeding the stadium kitchen

**In one sentence: batch size is how many examples one step processes together, and throughput (samples per second) stays flat on CPU but rises steeply on GPU — 14.5x at batch 2048, only 0.67x (a CPU win) at batch 32.**

Why: the CPU has few cores, so 32 or 2048 examples take proportionally longer — samples/sec barely moves (60000 → 65000 → 62000). The GPU has thousands of cores: at batch 32 most sit idle while fixed launch + transfer costs dominate (40000 samples/sec — slower than CPU!), at 256 they wake up (300000), at 2048 they all chop at once (900000). The lab sweeps exactly these three sizes so you see all three regimes. Watch it: the flat-CPU vs rising-GPU curves with both ratios printed.

In [ ]:
import numpy as np  # reuse numpy for the throughput arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the scaling plot
np.random.seed(3)  # fix the seed (numbers below are fixed; seed keeps the habit)
batches = np.array([32, 256, 2048])  # define the lab sweep: tiny, medium, huge
cpu_sps = np.array([60000.0, 65000.0, 62000.0])  # CPU throughput: flat (few cores, proportional time)
gpu_sps = np.array([40000.0, 300000.0, 900000.0])  # GPU throughput: starved, waking, saturated
print('CPU flat: 60000 -> 65000 -> 62000 (ratio max/min', round(float(cpu_sps.max()/cpu_sps.min()), 2), ')')  # print 1.08: barely moves
print('GPU rises: 40000 -> 300000 -> 900000 (ratio max/min', round(float(gpu_sps.max()/gpu_sps.min()), 1), ')')  # print 22.5: saturates
print('speedup at 2048:', round(float(gpu_sps[2]/cpu_sps[2]), 1), 'x; at 32:', round(float(gpu_sps[0]/cpu_sps[0]), 2), 'x')  # print 14.5x win, 0.67x loss
gain = gpu_sps / cpu_sps  # elementwise GPU-over-CPU speedup per batch size
fig, ax = plt.subplots(figsize=(7, 4))  # create one figure with one axes
ax.plot(batches, cpu_sps, 'o-', label='CPU (few fast cores)')  # draw the flat CPU line
ax.plot(batches, gpu_sps, 's-', label='GPU (thousands of small cores)')  # draw the rising GPU line
ax.set_xscale('log')  # log x because batches span 32 to 2048
ax.set_xlabel('batch size (log scale)')  # label x as batch size
ax.set_ylabel('samples/sec (higher is better)')  # label y as throughput
ax.set_title('CPU-flat vs GPU-rising: the GPU wins when fed (14.5x at 2048)')  # title the saturation story
for b, g in zip(batches, gain):  # annotate each batch size with its speedup
    ax.text(b, 950000, f'{g:.1f}x', ha='center', fontsize=9)  # speedup label above the GPU point
ax.legend()  # show which line is which device
ax.grid(True, which='both', alpha=0.3)  # faint grid on both scales
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure inline

### §6 Precisions and Tensor Cores — fewer bits, same answers

**In one sentence: FP32 spends 4 bytes per number while BF16 spends 2 (same values, fewer bits), so BF16 roughly halves weight and activation memory — and on modern GPUs it additionally unlocks Tensor Cores, special units that crunch half-precision matrix math extra fast.**

The lab's MLP holds 203530 trainable numbers (784×256 + 256 + 256×10 + 10): 0.814 MB in FP32 vs 0.407 MB in BF16. One large batch of hidden activations (2048×256) costs 2.097 MB in FP32 vs 1.049 MB in BF16. The number-line sketch shows the idea: FP32 spreads many fine-grained values across 32 bits, BF16 keeps the same range (same exponents, so big/small values survive) with a coarser mantissa (less fine detail — harmless for this MLP's accuracy). Risk: rounding error on sensitive models — but here accuracy stays about the same while memory halves and Tensor Cores speed the math.

> FP16 (missing from the lab sweeps, which cover FP32/BF16): same 2 bytes as BF16 but with a *narrower exponent range* — big/small activations overflow to inf/0 where BF16 survives. Rule of thumb: BF16 trains safely without loss-scaling; FP16 needs loss-scaling on sensitive models. That range advantage is why the lab sweeps BF16, not FP16. Watch it: the format number-line plus the four memory bars.

In [ ]:
import numpy as np  # reuse numpy for the memory-size math
import matplotlib.pyplot as plt  # reuse pyplot for the number-line + memory bars
np.random.seed(4)  # fix the seed (no randomness below; seed keeps the habit)
n_in, hidden = 784, 256  # define MLP shape matching the lab (28x28 in, 256 hidden)
params = n_in * hidden + hidden + hidden * 10 + 10  # count scalars: W1 + b1 + W2 + b2 = 203530
print('parameters:', params)  # print 203530: the total trainable scalar count
fp32_bytes = params * 4  # weight bytes in FP32 (4 bytes per number)
bf16_bytes = params * 2  # weight bytes in BF16 (2 bytes per number)
print('weights FP32 MB:', round(fp32_bytes / 1e6, 3))  # print 0.814: FP32 weight memory
print('weights BF16 MB:', round(bf16_bytes / 1e6, 3))  # print 0.407: BF16 halves it
batch = 2048  # define a large batch where activation memory matters
act_fp32 = batch * hidden * 4  # one hidden-activation batch in FP32 bytes
act_bf16 = batch * hidden * 2  # the same activations in BF16 bytes
print('activations FP32 MB:', round(act_fp32 / 1e6, 3))  # print 2.097: FP32 activation memory
print('activations BF16 MB:', round(act_bf16 / 1e6, 3))  # print 1.049: BF16 activation memory
print('Tensor Cores: half-precision matrix units -> extra speedup beyond memory savings')  # print the bonus reason
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))  # two panels: number-line + memory bars
ax[0].hlines([1, 0], 0, 32, colors=['steelblue', 'darkorange'])  # draw two 32-wide lines: FP32 vs BF16 bit rows
ax[0].text(16, 1.15, 'FP32: 1 sign + 8 exp + 23 mantissa (fine detail)', ha='center', fontsize=8)  # FP32 layout label
ax[0].text(16, 0.15, 'BF16: 1 sign + 8 exp + 7 mantissa (same range, coarser)', ha='center', fontsize=8)  # BF16 layout label
ax[0].set_xlim(0, 32)  # fix bit axis 0..32
ax[0].set_ylim(-0.5, 1.8)  # fix vertical room for both rows
ax[0].set_xlabel('bits')  # label the bit axis
ax[0].set_title('Format number-line: BF16 keeps range, trims detail')  # left title
ax[0].set_yticks([])  # hide y ticks: only bit positions matter
labels = ['weights fp32', 'weights bf16', 'activ batch fp32', 'activ batch bf16']  # four bar labels
vals = [fp32_bytes / 1e6, bf16_bytes / 1e6, act_fp32 / 1e6, act_bf16 / 1e6]  # four byte counts in megabytes
ax[1].bar(labels, vals, color=['steelblue', 'darkorange', 'steelblue', 'darkorange'])  # draw the memory bars
ax[1].set_ylabel('megabytes (lower is better)')  # label y as memory size
ax[1].set_title('BF16 uses about half the memory (0.814->0.407, 2.097->1.049)')  # right title with the numbers
fig.autofmt_xdate(rotation=15)  # tilt x labels so long names do not overlap
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §7 H2D transfer and synchronize — timing honestly

**In one sentence: H2D (host-to-device) transfer copies each batch from CPU memory to GPU memory and is a real cost the lab includes on purpose, while `synchronize()` blocks until queued GPU work actually finishes — without it your stopwatch measures how fast you gave the order, not how fast the kitchen cooked.**

GPU launches are asynchronous (fire-and-forget): the CPU queues a kernel and moves on, so timing without sync reports a 0.5 ms lie for 5 ms of real work. The lab's harness syncs before starting the timer and after the work, so the measured 3 ms H2D + 2 ms compute = 5 ms per step is honest. H2D matters most at small batches: moving 0.013 MB (batch 32) costs nearly as much as moving it as computing it, which is half of why batch 32 looks CPU-competitive (§8 is the other half). Watch it: the transfer/compute timeline showing the lie vs the truth.

In [ ]:
import numpy as np  # reuse numpy for the timeline arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the transfer/compute timeline
np.random.seed(5)  # fix the seed (numbers below are fixed; seed keeps the habit)
h2d, compute = 3.0, 2.0  # honest per-step costs in ms: 3 ms transfer + 2 ms compute
honest = h2d + compute  # honest synced time: 5.0 ms per step
lie = 0.5  # unsynced measured time in ms: only the queueing cost, the 5 ms lie
print('honest per-step time:', honest, 'ms (3 ms H2D + 2 ms compute)')  # print 5.0: the synced truth
print('unsynced stopwatch reports:', lie, 'ms (async lie: measures queueing, not work)')  # print 0.5: the lie
print('batches/sec honest:', round(1000.0 / honest), '(vs fake', round(1000.0 / lie), 'unsynced)')  # print 200 vs 2000
steps = 10  # draw 10 steps of the steady-state timeline
fig, ax = plt.subplots(figsize=(8, 3.2))  # make a wide flat figure for the timeline
ax.set_xlim(0, honest * steps)  # fix x-range to the honest total time
ax.set_ylim(0, 3)  # fix y-range for the two lanes
for s in range(steps):  # draw one transfer+compute pair per step
    ax.barh(1.6, h2d, left=s * honest, height=0.6, color='wheat')  # wheat block: H2D transfer chunk
    ax.barh(0.7, compute, left=s * honest + h2d, height=0.6, color='steelblue')  # blue block: compute chunk
ax.text(honest * steps / 2, 2.6, 'wheat = H2D transfer (3 ms), blue = compute (2 ms), sync at each edge', ha='center', fontsize=9)  # legend line
ax.set_xlabel('time in ms (synced: 5 ms per step x 10 steps = 50 ms)')  # label x with the honest total
ax.set_yticks([])  # hide y ticks: lanes are labeled by color
ax.set_title('Timeline: transfer + compute with sync (the 0.5 ms unsynced lie skipped)')  # title the honesty rule
plt.tight_layout()  # snug the layout
plt.show()  # render the timeline

### §8 Under-utilization and VRAM — what small batches waste

**In one sentence: under-utilization means too-small batches leave most GPU cores idle while fixed per-kernel launch costs dominate, and VRAM (peak GPU megabytes) tells you how much headroom a bigger batch or a narrower dtype would buy.**

> Numbers in this lecture (CPU ~60k, GPU 40k→900k, 14.5×) are an *illustrative sketch* — your `notebook.ipynb` measures the real ones on your machine (CPU-only boxes produce no GPU rows). The *shape* (flat-then-rising GPU curve) is the claim.

Picture 100 GPU workers: at batch 32 only about 2 are chopping while 98 wait — yet every kernel still pays its ~microsecond launch fee, so overhead, not math, sets the runtime. At batch 2048 all 100 chop and the fee spreads thin. VRAM makes the same point in memory: the lab records peak megabytes per config, and BF16 roughly halves every bar — the freed headroom is exactly what lets you double the batch and climb toward saturation (§5). Watch it: the active-vs-idle worker bars plus the VRAM bars per batch and dtype.

In [ ]:
import numpy as np  # reuse numpy for the utilization + VRAM math
import matplotlib.pyplot as plt  # reuse pyplot for the waste bars
np.random.seed(6)  # fix the seed (numbers below are fixed; seed keeps the habit)
active32, active2048 = 2, 100  # active workers of 100: batch 32 idles 98, batch 2048 saturates
print('batch 32: 2/100 workers busy (98 idle) -> launch + H2D dominate')  # print the starvation line
print('batch 2048: 100/100 workers busy -> math dominates, GPU 14.5x')  # print the saturation line
vram = {'b32 fp32': 12.0, 'b32 bf16': 6.0, 'b2048 fp32': 410.0, 'b2048 bf16': 205.0}  # sketch peak VRAM MB per config
print('VRAM b2048 fp32 -> bf16:', vram['b2048 fp32'], '->', vram['b2048 bf16'], 'MB (halved)')  # print the headroom win
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: workers + VRAM
ax[0].bar(['batch 32\nactive', 'batch 32\nidle', 'batch 2048\nactive'], [2, 98, 100], color=['red', 'lightgray', 'green'])  # worker bars
ax[0].set_ylabel('workers of 100')  # label y as worker count
ax[0].set_title('Small batch starves the machine (2 busy, 98 idle)')  # left title
ax[1].bar(list(vram.keys()), list(vram.values()), color=['steelblue', 'darkorange', 'steelblue', 'darkorange'])  # VRAM bars
ax[1].set_ylabel('peak VRAM MB (lower is better)')  # label y as megabytes
ax[1].set_title('VRAM: BF16 halves every bar (410 -> 205)')  # right title with the numbers
fig.autofmt_xdate(rotation=15)  # tilt labels so long names fit
ax[0].grid(True, axis='y', alpha=0.3)  # faint grid left
ax[1].grid(True, axis='y', alpha=0.3)  # faint grid right
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

## How it all fits together — the system map §9

One benchmark point is a pipeline with every § term playing a position. **Pick device+dtype** (§4, §6): build the MLP directly in `cuda/BF16`. **Feed batches** (§5): the DataLoader deals 32, 256, or 2048 slices — the batch-size arrow below pushes the point rightward and upward. **Copy H2D** (§7): each slice crosses the wire (honestly timed). **Compute** (§5, §6): thousands of cores plus Tensor Cores do the math. **Sync + measure** (§7): `synchronize()` brackets the stopwatch; throughput = samples ÷ seconds. **Diagnose** (§8): flat curve + tiny-batch loss means starvation (idle workers, launch overhead), fat VRAM bars mean switch dtype or grow the batch. The map below pins each term to its station; the numbers time one honest row.

In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.8))  # wide figure for the benchmark pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('CPU batch\n§5', 1.2), ('H2D copy\n§7', 3.8), ('GPU compute\n§5+§6', 6.4), ('sync\n§7', 9.0), ('measure\nsps §5', 11.4), ('diagnose\n§8', 13.2)]  # pipeline stations in order
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.5, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = measurement pipeline
for xa, xb in [(1.9, 3.0), (4.6, 5.6), (7.4, 8.4), (9.6, 10.6), (12.0, 12.6)]:  # gaps between stations
    ax.annotate('', xy=(xb, 2.5), xytext=(xa, 2.5), arrowprops=dict(arrowstyle='->'))  # forward arrows left to right
ax.annotate('', xy=(12.5, 1.2), xytext=(1.2, 1.2), arrowprops=dict(arrowstyle='->', color='green', lw=2))  # green batch-size arrow below
ax.text(7, 3.6, 'ONE BENCHMARK ROW: copy -> compute -> sync -> score (bigger batch = fuller GPU)', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(7, 0.5, 'batch-size arrow: 32 (starved) -> 256 -> 2048 (saturated, 14.5x)', ha='center', fontsize=9, fontweight='bold', color='darkgreen')  # green = saturation lesson
ax.set_title('System map §9: one (device, dtype, batch) point through every term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the one-row timing trace
np.random.seed(9)  # fix the seed (numbers below are fixed; habit kept)
dataset, batch = 60000, 256  # trace setup: full dataset with a medium batch size
per_epoch = (dataset + batch - 1) // batch  # ceil division: 60000/256 -> 235 slice-steps per epoch
epoch_s = 1.0  # honest synced epoch time in seconds (H2D + compute + sync included)
sps = dataset / epoch_s  # throughput = samples divided by seconds = 60000.0
print('steps per epoch: ceil(60000/256) =', per_epoch)  # print 235: the DataLoader slice count
print('epoch time (synced):', epoch_s, 's -> throughput:', sps, 'samples/sec')  # print 60000.0: the headline metric
print('unsynced lie would claim ~10x more (measures queueing, §7)')  # print why sync brackets matter
print('VRAM says: BF16 halves this row, headroom for batch 2048 (§8)')  # print the diagnose step

## Bridge to the lab

Open `notebook.ipynb` now. It measures the sketch above for real (CPU rows here; GPU rows appear on a CUDA machine):

1. **Cell 1 (goal markdown)** — the question: why does a GPU accelerate ML? Answer it with the stadium-kitchen picture plus numbers (§5).
2. **Cell 2 (setup + detect)** — seeds, `results/` folder, and the `HAS_CUDA` / `DEV` device pick (§4). GPU property prints only run when CUDA exists.
3. **Cell 3 (data)** — same MNIST-or-digits fallback, kept on CPU and moved per batch (so H2D cost is honestly included, §7).
4. **Cell 4 (model + `train_config`)** — the same MLP built directly in the target dtype/device (§4, §6); the harness warms up, times with `synchronize()` (§7), and records peak VRAM (§8).
5. **Cell 5 (the sweep)** — batches [32, 256, 2048] on CPU/FP32, plus GPU × (FP32, BF16) when available (§5, §6); every row lands in `results/gpu_lab.csv` with time, samples/sec, peak MB. This cell produces the rising-GPU / flat-CPU curve (14.5x at 2048, 0.67x at 32).
6. **Cell 6 (throughput plot)** — reads the CSV back and draws throughput vs batch size (log x). Expect: GPU ≫ CPU at 256+, CPU competitive at 32 (§8 explains why).
7. **Cell 7 (README prompts)** — when the GPU does NOT win (tiny batch, transfer-bound, §7–§8), and what BF16 changed (memory ≈ halved to 0.407/1.049 MB, similar accuracy, §6).

## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Why can a CPU beat a GPU at batch size 32 but lose 14.5x at 2048?**

*Answer: at 32 most GPU workers sit idle (2 of 100 busy, §8) while fixed kernel-launch and H2D-transfer costs dominate (§7); at 2048 the work saturates thousands of cores and high-bandwidth memory, so throughput scales (§5) and the CPU is left behind.*

2. **What does `torch.cuda.synchronize()` do, and why must it wrap GPU timing?**

*Answer: it blocks until queued GPU kernels actually finish (§7). Launches are asynchronous (fire-and-forget), so timing without sync reports the 0.5 ms queueing lie instead of the honest 5.0 ms — the timeline above shows both.*

3. **What do you gain (and risk) by training in BF16 instead of FP32?**

*Answer: you roughly halve weight/activation memory (0.814 → 0.407 MB, 2.097 → 1.049 MB) and traffic, plus Tensor-Core speed (§6); you risk rounding error, but on this MLP accuracy stays about the same — the bars show the memory win and VRAM (§8) buys headroom for bigger batches.*